# Football Player Tracker | CVAT MOT Tracking and Batch Export

Use this Google Colab notebook to generate **CVAT MOT 1.1 tracking ZIP files**, optional tracked-preview videos, and batch result summaries for American football clips. Tracking code and BoT-SORT configuration come from the [`Football-Player-Tracker` GitHub repository](https://github.com/Alejandro-Fuste/Football-Player-Tracker); input clips and exports are stored in Google Drive.

## Before you begin

- Select a **GPU runtime** in Colab (recommended).
- Make sure your GitHub `main` branch contains the BoxMOT 25-compatible tracker implementation and `gmc_method: sof`. **Cell D** checks these requirements.
- Put the source videos in your configured Google Drive clips directory. Provide a `ToBeTracked.csv` with a **`filename`** or **`shortname`** column containing video filenames or stems.
- Confirm your detector weights and ReID weights are accessible; **Cell D** checks model paths.

## Run order

**Fresh Colab session:** Run **A** → select **Runtime → Restart session** → run **B, C, D, E, F** → choose **G (additional batch check)** or **H (production CSV batch)**. Cell H is the preferred production workflow. You do **not** need to run G before H if F succeeds.

**Repeat batch in an initialized session:** Update your manifest and the settings in **H**, then run **H** again. Cell H explicitly selects its manifest source; it does not depend on Cell E's earlier CSV selection.

## Outputs and workflow roles

- **F — Smoke test:** Tests a single clip for five frames and writes a test MOT ZIP.
- **G — Basic batch check:** Uses the `CSV_PATH` selected by Cell E and writes to the `Reclipped_Tracking` directory. Optional; not the primary production exporter.
- **H — Production CSV batch:** Selects the current `ToBeTracked.csv` from **Drive or GitHub**, validates every requested clip, writes CVAT ZIPs and optional preview MP4s to the **`output_dir` specified in Cell H**, and audits the batch summary.

**Before every production run:** Review `MANIFEST_SOURCE`, `OVERWRITE`, `SAVE_PREVIEWS`, `MAX_VIDEOS`, `MAX_FRAMES`, `WEIGHTS_OVERRIDE`, and `output_dir` in **H**. With `OVERWRITE = False`, existing ZIPs are skipped; for changed/reclipped clips with the same names, use `True` when you intend to regenerate them. Existing files in an output directory may remain from previous runs; inspect the batch summary for the current run.

> The executable cells are preserved from the successfully tested uploaded notebook. This edition updates explanations and section titles only.


## A — Install dependencies (once per fresh runtime)

Installs the specified package versions for tracking and export. **After this cell finishes, restart the Colab runtime** using **Runtime → Restart session** so Python loads a consistent set of compiled NumPy/SciPy modules. Then continue with **B**; do not rerun A during the same setup unless you intentionally reinstall the environment.


In [7]:
import sys, subprocess

# Keep NumPy/SciPy in the same dependency transaction as BoxMOT and Ultralytics.
# Avoid unconstrained `pip install scipy boxmot ...` and repeated in-notebook upgrades.
packages = [
    'numpy==2.3.3',
    'scipy==1.16.2',
    'boxmot==25.0.0',
    'ultralytics==8.4.183',
    'opencv-python-headless',
    'pandas', 'pyyaml', 'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                '--upgrade-strategy', 'only-if-needed', *packages], check=True)
# Replace any mixed NumPy/SciPy Python/binary files (without disturbing Torch).
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                '--force-reinstall', '--no-deps', '--only-binary=:all:',
                'numpy==2.3.3', 'scipy==1.16.2'], check=True)

# Verify the *on-disk* packages in a fresh process, not cached notebook imports.
probe = "import numpy as n, scipy as s, scipy.linalg; print('NumPy',n.__version__,'SciPy',s.__version__); assert n.__version__=='2.3.3' and s.__version__=='1.16.2'; print('det',scipy.linalg.det(n.eye(2)))"
subprocess.run([sys.executable, '-c', probe], check=True)
print('INSTALL COMPLETE. REQUIRED: Runtime > Restart session. Then continue at Cell B.')


INSTALL COMPLETE. REQUIRED: Runtime > Restart session. Then continue at Cell B.


## B — Verify the installed environment

Run **after restarting**. Confirms NumPy, SciPy, OpenCV, and BoxMOT can import correctly in a fresh Python subprocess. Do not continue to tracking until this verification passes.


In [8]:
import sys, subprocess, importlib.metadata
probe = r"""
import numpy as np
import scipy
import scipy.linalg
import cv2
from boxmot.trackers.box.botsort.tracker import BotSort
assert np.__version__ == '2.3.3', np.__version__
assert scipy.__version__ == '1.16.2', scipy.__version__
print('NumPy',np.__version__,'SciPy',scipy.__version__)
print('SciPy det',scipy.linalg.det(np.eye(2)))
print('OpenCV',cv2.__version__, 'BotSort import successful')
"""
subprocess.run([sys.executable, '-c', probe], check=True)
print('All compiled imports passed. No need to reinstall or restart again.')


All compiled imports passed. No need to reinstall or restart again.


## C — Mount Google Drive and synchronize GitHub source

Connects Drive, defines `PROJECT_CODE_DIR`, `DRIVE_PROJECT_DIR`, `CLIPS_DIR`, `EXPORT_DIR`, and `CSV_PATH`, and clones or refreshes the GitHub `main` branch. **GitHub is authoritative for Python code and tracker configuration; Drive holds video inputs and exports.**

**Note:** The existing code executes `git reset --hard origin/main` on the Colab checkout. Local, uncommitted changes in that Colab clone will be discarded. Commit intended source changes to GitHub first.


In [9]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os
DRIVE_ROOT = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy')
DRIVE_PROJECT_DIR = DRIVE_ROOT / 'Utilities' / 'Football-Player-Tracker'
PROJECT_CODE_DIR = Path('/content/Football-Player-Tracker')
REPO_URL = 'https://github.com/Alejandro-Fuste/Football-Player-Tracker.git'
CLIPS_DIR = DRIVE_ROOT / 'Datasets' / 'AmericanFootballDataset' / 'videos'
EXPORT_DIR = DRIVE_PROJECT_DIR / 'exports'
CSV_PATH = DRIVE_PROJECT_DIR / 'ToBeTracked.csv'
if not DRIVE_PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Create Drive output directory: {DRIVE_PROJECT_DIR}')
if PROJECT_CODE_DIR.exists():
    subprocess.run(['git','-C',str(PROJECT_CODE_DIR),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(PROJECT_CODE_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','--branch','main',REPO_URL,str(PROJECT_CODE_DIR)],check=True)
commit = subprocess.check_output(['git','-C',str(PROJECT_CODE_DIR),'rev-parse','HEAD'],text=True).strip()
os.environ['FOOTBALL_TRACKER_EXPORT_DIR']=str(EXPORT_DIR)
os.chdir(PROJECT_CODE_DIR)
if str(PROJECT_CODE_DIR) not in sys.path: sys.path.insert(0,str(PROJECT_CODE_DIR))
print('GitHub commit:',commit)
print('Source:',PROJECT_CODE_DIR,'\nVideos:',CLIPS_DIR,'\nExports:',EXPORT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GitHub commit: 28188ded36b7faf1d6258e4ba46841d0375c2b39
Source: /content/Football-Player-Tracker 
Videos: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/AmericanFootballDataset/videos 
Exports: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football-Player-Tracker/exports


## D — Validate repository, tracker configuration, and model files

Checks that the GitHub checkout contains the supported BoxMOT tracker integration, has no legacy path-injection code, and uses `gmc_method: sof`. Also checks the configured Drive export path and the local ReID weight file, and lists detector weight locations.

This cell **checks** source compatibility; it does not modify Python files. If a check fails, update and push the GitHub repository, then rerun **C** and **D**.


In [10]:
import ast, yaml, os
tracker_path = PROJECT_CODE_DIR/'src/tracker.py'
script_path = PROJECT_CODE_DIR/'scripts/detect_track_export.py'
project_config_path = PROJECT_CODE_DIR/'configs/project_config.py'
tracker_yaml = PROJECT_CODE_DIR/'configs/football_botsort.yaml'
for p in [tracker_path, script_path, project_config_path, tracker_yaml]:
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix == '.py': ast.parse(p.read_text())
failures=[]
if '/tmp/boxmot_repo' in script_path.read_text():
    failures.append('scripts/detect_track_export.py still contains legacy BoxMOT import forcing')
if 'from boxmot.trackers.box.botsort.tracker import BotSort' not in tracker_path.read_text():
    failures.append('src/tracker.py has not been migrated to the BoxMOT 25 direct BotSort adapter')
if 'FOOTBALL_TRACKER_EXPORT_DIR' not in project_config_path.read_text():
    failures.append('configs/project_config.py needs GitHub-source/Drive-export migration')
cfg=yaml.safe_load(tracker_yaml.read_text())
if cfg.get('gmc_method') != 'sof':
    failures.append(f"configs/football_botsort.yaml has gmc_method={cfg.get('gmc_method')!r}, should be 'sof'")
if failures:
    raise RuntimeError('GitHub main requires migration; apply migrate_boxmot25.py locally, commit and push, then rerun Cell C:\n - '+'\n - '.join(failures))
from configs import project_config
assert project_config.PROJECT_ROOT.resolve() == PROJECT_CODE_DIR.resolve()
assert project_config.EXPORT_DIR.resolve() == EXPORT_DIR.resolve()
assert CLIPS_DIR.is_dir(), f'Videos directory not found: {CLIPS_DIR}'
assert (PROJECT_CODE_DIR/'models/osnet_x0_25_msmt17.pt').is_file()
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
print('GitHub tracker and configuration passed. Camera motion:',cfg['gmc_method'])
registry=yaml.safe_load((PROJECT_CODE_DIR/'configs/model_registry.yaml').read_text())
for name, spec in registry.get('models',{}).items():
    p=Path(spec.get('weights_path',''))
    print('Detector model:',name,'weights exist:',p.is_file(),p)


GitHub tracker and configuration passed. Camera motion: sof
Detector model: rtdetrx_player_best weights exist: True /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/Object_Detection/Models/YOLO/Output/RTDETR/weights/best.pt
Detector model: yolo26x_player_best weights exist: True /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/Object_Detection/Models/YOLO/Output/YOLO26/weights/best.pt


## E — Validate CSV inputs and initialize BoT-SORT

Reads `CSV_PATH` (Drive `ToBeTracked.csv`, falling back to the repository copy if the Drive file is absent), resolves each listed clip against `CLIPS_DIR`, and initializes the tracker with an empty detection array and a real video frame.

This is a **preflight check**, not a batch export. **Cell H selects its own manifest independently**, so confirm Cell H's `MANIFEST_SOURCE` for production runs.


In [11]:
import csv, cv2, numpy as np
from src.tracker import create_botsort_tracker
if not CSV_PATH.is_file():
    CSV_PATH = PROJECT_CODE_DIR/'ToBeTracked.csv'
if not CSV_PATH.is_file(): raise FileNotFoundError('ToBeTracked.csv missing in Drive or GitHub repo')
with CSV_PATH.open(newline='',encoding='utf-8-sig') as f:
    reader=csv.DictReader(f)
    field=next((k for k in (reader.fieldnames or []) if k.strip().lower() in ('filename','shortname')),None)
    if field is None: raise ValueError('CSV needs FileName or ShortName header')
    names=[row[field].strip() for row in reader if row.get(field,'').strip()]
if not names: raise ValueError('Manifest is empty')
def find_clip(name):
    p=CLIPS_DIR/name
    if p.is_file(): return p
    for ext in ('.mp4','.MP4','.mov','.MOV','.avi','.AVI','.mkv','.MKV'):
        q=CLIPS_DIR/(Path(name).stem+ext)
        if q.is_file(): return q
    raise FileNotFoundError(f'Missing clip: {name}')
missing=[]
for name in names:
    try: find_clip(name)
    except FileNotFoundError: missing.append(name)
if missing: raise FileNotFoundError(f'Missing {len(missing)} video(s): {missing[:20]}')
print('Manifest:',CSV_PATH,'\nClip count:',len(names))
first_path=find_clip(names[0]); cap=cv2.VideoCapture(str(first_path))
try: ok,frame=cap.read()
finally: cap.release()
if not ok: raise RuntimeError(f'Cannot read video frame: {first_path}')
tracker=create_botsort_tracker(tracker_yaml)
out=tracker.update(np.empty((0,6),dtype=np.float32),frame)
print('BoT-SORT empty-frame update:',np.asarray(out).shape)


Manifest: /content/Football-Player-Tracker/ToBeTracked.csv 
Clip count: 4


INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

[FootballMOT] BotSort: device=cuda:0, ReID=True, CMC=sof
BoT-SORT empty-frame update: (0, 8)


## F — Five-frame detector-to-CVAT smoke test

Runs the detector and tracker on a short sample from the first selected clip, writes a CVAT MOT ZIP, and verifies the export path. This is the recommended final check before executing a full batch.


In [5]:
import subprocess,sys,zipfile
SMOKE_OUT=EXPORT_DIR/'cvat_mot_tracks'/'smoke_test'/f'{first_path.stem}_cvat_mot.zip'
SMOKE_OUT.parent.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,'scripts/detect_track_export.py',
    '--video',str(first_path),'--model-config',str(PROJECT_CODE_DIR/'configs/model_registry.yaml'),
    '--model-key','rtdetrx_player_best','--tracker-config',str(tracker_yaml),
    '--out',str(SMOKE_OUT),'--conf','0.64','--max-frames','5']
subprocess.run(cmd,check=True,cwd=PROJECT_CODE_DIR,env=os.environ.copy())
assert SMOKE_OUT.is_file() and SMOKE_OUT.stat().st_size>0
with zipfile.ZipFile(SMOKE_OUT) as z:
    print('MOT ZIP members:',z.namelist())
    assert any(p.endswith('gt.txt') for p in z.namelist())
print('5-frame smoke test passed:',SMOKE_OUT)


MOT ZIP members: ['gt/gt.txt', 'gt/labels.txt']
5-frame smoke test passed: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football-Player-Tracker/exports/cvat_mot_tracks/smoke_test/JetSweep_66_cvat_mot.zip


## G — Optional basic batch check

Runs the repository's batch exporter using the `CSV_PATH` selected in **E**. Outputs are placed under `exports/cvat_mot_tracks/batch/Reclipped_Tracking` and a batch summary is inspected for failures.

**Optional:** This is useful for a small regression check but does not expose the production configuration controls available in **H**. Running G is **not required** before running H.


In [6]:
import json
BATCH_NAME='Reclipped_Tracking'
BATCH_OUT=EXPORT_DIR/'cvat_mot_tracks'/'batch'/BATCH_NAME
cmd=[sys.executable,'scripts/batch_export.py',
    '--model-config',str(PROJECT_CODE_DIR/'configs/model_registry.yaml'),
    '--model-key','rtdetrx_player_best','--tracker-config',str(tracker_yaml),
    '--clips-dir',str(CLIPS_DIR),'--out-dir',str(BATCH_OUT),
    '--csv-list',str(CSV_PATH),'--conf','0.64','--save-preview','--write-summary']
result=subprocess.run(cmd,cwd=PROJECT_CODE_DIR,env=os.environ.copy())
summary_path=BATCH_OUT/'batch_tracking_summary.json'
if not summary_path.is_file(): raise RuntimeError(f'Batch summary missing; exit={result.returncode}')
items=json.loads(summary_path.read_text())
for row in items: print(row.get('video_name',row.get('video')),row.get('status'),row.get('error_message',''))
failed=[row for row in items if row.get('status')=='failed']
if failed:
    for row in failed:
        log=BATCH_OUT/'logs'/(Path(row.get('video_name','unknown')).stem+'_error.log')
        if log.exists(): print(log.read_text()[-6000:])
    raise RuntimeError(f'Batch has {len(failed)} failed clip(s).')
if result.returncode: raise RuntimeError(f'Batch process exit code {result.returncode}')
print('Batch completed. Entries:',len(items),'Output:',BATCH_OUT)


JetSweep_66.mp4 success 
JetSweep_67.mp4 success 
JetSweep_68.mp4 success 
JetSweep_69.mp4 success 
Batch completed. Entries: 4 Output: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football-Player-Tracker/exports/cvat_mot_tracks/batch/Reclipped_Tracking


## H — Production batch export from `ToBeTracked.csv`

**Use this cell for full or resumed production runs.** It explicitly loads the requested CSV, validates its entries and input video files, invokes the repository batch exporter, and checks the resulting summary and expected CVAT ZIP files.

### Configure before running

1. Set **`MANIFEST_SOURCE`** to `"drive"` (for `DRIVE_PROJECT_DIR/ToBeTracked.csv`) or `"github"` (for the latest CSV on remote GitHub `main`). This avoids accidentally using an outdated manifest.
2. Review **`output_dir`** inside the cell. The current code writes under `exports/cvat_mot_tracks/batch/BootPass/reclipped`. Change that setting in the code only when you intentionally choose a different batch destination.
3. Review **`OVERWRITE`** (`False` skips existing ZIPs; `True` reprocesses existing filenames), **`SAVE_PREVIEWS`**, **`MAX_VIDEOS`**, and **`MAX_FRAMES`** (`-1` processes whole videos).
4. Confirm detector settings **`CONFIDENCE`**, **`IMAGE_SIZE`**, **`MODEL_KEY`**, and **`WEIGHTS_OVERRIDE`** match your validated run.
5. Run H and confirm the printed **SELECTED MANIFEST** and **VIDEOS READ FROM ToBeTracked.csv** match your intended clips before interpreting the results.

### Expected results

The cell reports successful, skipped, and failed videos; checks that the batch summary matches the requested names; and verifies corresponding `_cvat_mot.zip` exports. If enabled, preview MP4 files are written in the batch output's `previews/` subfolder. Per-clip diagnostics and summary files are written under the batch output directory.

**Important:** The code can generate a successful batch without deleting older preview videos. Use the summary and listed ZIP names to confirm what was processed in the current execution.


In [13]:

from pathlib import Path
import csv
import json
import os
import subprocess
import sys

# ============================================================
# 1. SETTINGS
# ============================================================

# Choose where YOUR UPDATED ToBeTracked.csv is stored:
# "drive"  = Google Drive project folder
# "github" = latest origin/main in GitHub
MANIFEST_SOURCE = "drive"

# Set True when reclipped videos must replace old tracks.
OVERWRITE = False
SAVE_PREVIEWS = True

# None = every video in the CSV
MAX_VIDEOS = None

# -1 = process every frame
MAX_FRAMES = -1

# Match the settings used by successful Cell G.
CONFIDENCE = 0.64
IMAGE_SIZE = 640
MODEL_KEY = "rtdetrx_player_best"

# Set this if Cell G used explicit detector weights.
WEIGHTS_OVERRIDE = None

# ============================================================
# 2. PROJECT AND OUTPUT PATHS
# ============================================================

project = Path(PROJECT_CODE_DIR).resolve()
clips_dir = Path(CLIPS_DIR).resolve()

drive_project = Path(
    globals().get(
        "DRIVE_PROJECT_DIR",
        "/content/drive/MyDrive/UCF/Courses/Spring2026/"
        "IndependentStudy/Utilities/Football-Player-Tracker"
    )
)

# Keep batch outputs separate from the previous test run.
output_dir = (
    drive_project / "exports" / "cvat_mot_tracks" / "batch" / "BootPass" / "reclipped"
)
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# 3. LOAD THE EXPLICIT CSV SOURCE
# ============================================================

if MANIFEST_SOURCE == "drive":

    manifest = drive_project / "ToBeTracked.csv"

    if not manifest.is_file():
        raise FileNotFoundError(
            f"Drive manifest not found: {manifest}"
        )

elif MANIFEST_SOURCE == "github":

    subprocess.run(
        ["git", "-C", str(project), "fetch", "origin", "main"],
        check=True
    )

    # Read the CSV directly from the latest remote main.
    csv_content = subprocess.check_output(
        [
            "git", "-C", str(project),
            "show", "origin/main:ToBeTracked.csv"
        ],
        text=True
    )

    # Save a snapshot for the batch exporter.
    manifest = project / ".colab_batch_manifest.csv"
    manifest.write_text(csv_content)

else:
    raise ValueError(
        "MANIFEST_SOURCE must be 'drive' or 'github'."
    )

print("=" * 60)
print("SELECTED MANIFEST")
print("Source:", MANIFEST_SOURCE)
print("Path:", manifest)
print("=" * 60)

# ============================================================
# 4. READ AND VALIDATE CSV CONTENT
# ============================================================

with manifest.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    reader = csv.DictReader(f)

    if not reader.fieldnames:
        raise ValueError("ToBeTracked.csv is empty.")

    columns = {
        name.strip().lower(): name
        for name in reader.fieldnames
        if name
    }

    selected_column = (
        columns.get("filename")
        or columns.get("shortname")
    )

    if selected_column is None:
        raise ValueError(
            "ToBeTracked.csv must contain a "
            "'filename' or 'shortname' column. "
            f"Found: {reader.fieldnames}"
        )

    requested_names = [
        row[selected_column].strip()
        for row in reader
        if row.get(selected_column, "").strip()
    ]

# Remove duplicates without changing order.
requested_names = list(dict.fromkeys(requested_names))

if not requested_names:
    raise RuntimeError(
        "The selected CSV contains no video names."
    )

# ============================================================
# 5. DISPLAY EXACT VIDEOS TO BE PROCESSED
# ============================================================

print("\nVIDEOS READ FROM ToBeTracked.csv")
print("-" * 60)

for index, name in enumerate(requested_names, start=1):
    print(f"{index:03d}. {name}")

print("-" * 60)
print("Total requested:", len(requested_names))

# ============================================================
# 6. RESOLVE ALL VIDEO PATHS
# ============================================================

extensions = [
    ".mp4", ".MP4",
    ".mov", ".MOV",
    ".avi", ".AVI",
    ".mkv", ".MKV"
]

resolved_videos = []
missing_videos = []

for name in requested_names:

    given = Path(name)

    if given.is_absolute() or ".." in given.parts:
        raise ValueError(
            f"Unsafe video entry in manifest: {name}"
        )

    if given.suffix:
        candidates = [clips_dir / given]
    else:
        candidates = [
            clips_dir / f"{name}{ext}"
            for ext in extensions
        ]

    found = next(
        (p for p in candidates if p.is_file()),
        None
    )

    if found is None:
        missing_videos.append(name)
    else:
        resolved_videos.append(found)

if missing_videos:
    print("\nMISSING VIDEOS:")
    for name in missing_videos:
        print(" -", name)

    raise FileNotFoundError(
        f"{len(missing_videos)} requested videos "
        "were not found. Batch cancelled."
    )

# The repository batch exporter uses the basename/stem
# to generate CVAT ZIP names.
stems = [p.stem for p in resolved_videos]

if len(set(stems)) != len(stems):
    raise ValueError(
        "Multiple requested videos have the same filename "
        "stem, which would create conflicting export ZIPs."
    )

print("\nAll requested videos exist.")

# ============================================================
# 7. PREPARE BATCH COMMAND
# ============================================================

batch_script = project / "scripts" / "batch_export.py"
tracker_config = project / "configs" / "football_botsort.yaml"
model_config = project / "configs" / "model_registry.yaml"

for required in [
    batch_script,
    tracker_config,
    model_config
]:
    if not required.is_file():
        raise FileNotFoundError(required)

cmd = [
    sys.executable,
    str(batch_script),

    "--csv-list", str(manifest),
    "--clips-dir", str(clips_dir),
    "--out-dir", str(output_dir),

    "--model-config", str(model_config),
    "--model-key", MODEL_KEY,
    "--tracker-config", str(tracker_config),

    "--conf", str(CONFIDENCE),
    "--imgsz", str(IMAGE_SIZE),

    "--max-frames", str(MAX_FRAMES),
    "--write-summary",
]

if MAX_VIDEOS is not None:
    cmd.extend(["--max-videos", str(MAX_VIDEOS)])

if OVERWRITE:
    cmd.append("--overwrite")

if SAVE_PREVIEWS:
    cmd.append("--save-preview")

if WEIGHTS_OVERRIDE is not None:
    cmd.extend([
        "--weights",
        str(WEIGHTS_OVERRIDE)
    ])

expected_names = (
    resolved_videos[:MAX_VIDEOS]
    if MAX_VIDEOS is not None
    else resolved_videos
)

expected_stems = {p.stem for p in expected_names}

# ============================================================
# 8. RUN THE CSV BATCH
# ============================================================

print("\n" + "=" * 60)
print("STARTING CSV BATCH")
print("Manifest source:", MANIFEST_SOURCE)
print("Videos to process:", len(expected_names))
print("Output folder:", output_dir)
print("Overwrite:", OVERWRITE)
print("Save previews:", SAVE_PREVIEWS)
print("=" * 60)

result = subprocess.run(
    cmd,
    cwd=str(project),
    env=os.environ.copy(),
    check=False
)

if result.returncode != 0:
    raise RuntimeError(
        f"Batch process exited with code "
        f"{result.returncode}"
    )

# ============================================================
# 9. VALIDATE SUMMARY AGAINST THE CSV
# ============================================================

summary_path = (
    output_dir / "batch_tracking_summary.json"
)

if not summary_path.is_file():
    raise FileNotFoundError(
        f"Missing batch summary: {summary_path}"
    )

summary = json.loads(summary_path.read_text())

reported_stems = {
    Path(
        entry.get("video_name")
        or entry.get("video")
        or ""
    ).stem
    for entry in summary
}

unexpected = reported_stems - expected_stems
unreported = expected_stems - reported_stems

if unexpected or unreported:
    raise RuntimeError(
        "Batch summary does not match ToBeTracked.csv.\n"
        f"Unexpected videos: {sorted(unexpected)}\n"
        f"Missing videos: {sorted(unreported)}"
    )

counts = {
    "success": 0,
    "failed": 0,
    "skipped": 0
}

for entry in summary:
    status = entry.get("status", "unknown")
    if status in counts:
        counts[status] += 1

# ============================================================
# 10. REPORT RESULTS
# ============================================================

print("\n" + "=" * 60)
print("CSV BATCH RESULTS")
print("=" * 60)

print("Requested:", len(expected_stems))
print("Successful:", counts["success"])
print("Skipped:", counts["skipped"])
print("Failed:", counts["failed"])

for entry in summary:
    video_name = (
        entry.get("video_name")
        or entry.get("video")
    )

    print(
        f"{entry.get('status', 'unknown'):8s} "
        f"{video_name}"
    )

if counts["failed"]:
    for entry in summary:
        if entry.get("status") == "failed":
            print(
                "\nERROR:",
                entry.get("video_name"),
                entry.get("error_message", "")
            )

    raise RuntimeError(
        f"{counts['failed']} batch videos failed."
    )

print("\nExpected CVAT ZIPs:")

for stem in sorted(expected_stems):
    zip_path = output_dir / f"{stem}_cvat_mot.zip"
    print(
        "FOUND" if zip_path.is_file() else "MISSING",
        zip_path.name
    )

missing_zips = [
    stem
    for stem in expected_stems
    if not (
        output_dir / f"{stem}_cvat_mot.zip"
    ).is_file()
]

if missing_zips:
    raise RuntimeError(
        f"Missing CVAT MOT ZIPs for: {missing_zips}"
    )

print("\nBatch completed and outputs verified.")
print("Results folder:", output_dir)


SELECTED MANIFEST
Source: drive
Path: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football-Player-Tracker/ToBeTracked.csv

VIDEOS READ FROM ToBeTracked.csv
------------------------------------------------------------
001. BootPass_103.mp4
------------------------------------------------------------
Total requested: 1

All requested videos exist.

STARTING CSV BATCH
Manifest source: drive
Videos to process: 1
Output folder: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football-Player-Tracker/exports/cvat_mot_tracks/batch/BootPass/reclipped
Overwrite: False
Save previews: True

CSV BATCH RESULTS
Requested: 1
Successful: 1
Skipped: 0
Failed: 0
success  BootPass_103.mp4

Expected CVAT ZIPs:
FOUND BootPass_103_cvat_mot.zip

Batch completed and outputs verified.
Results folder: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football-Player-Tracker/exports/cvat_mot_tracks/batch/BootPass/reclipped
